# 00 — Viết qrels (bảng chân lý cho retrieval)

**qrels = bảng chân lý**: với mỗi câu hỏi, bài nào trả lời được, bài nào chỉ liên quan, bài nào không.

Không có qrels thì mọi metric (MRR, nDCG@k) chỉ là con số vô nghĩa. Đây là file bạn sẽ quay lại
nhiều lần: viết 12 query trước, sau đó mở rộng lên 40-50.

## Thang điểm (áp dụng NHẤT QUÁN cho mọi query)

| Điểm | Nghĩa | Cách tự hỏi |
|---|---|---|
| **2** | Bài **trả lời trực tiếp** câu hỏi | "Đọc xong bài này, người hỏi có biết đáp án không?" |
| **1** | **Liên quan một phần** — cùng chủ đề nhưng không trả lời trực tiếp | "Bài này nói về chuyện khác nhưng có nhắc tới khái niệm trong câu hỏi" |
| **0** | Không liên quan (chỉ chấm khi bạn đã thực sự xem bài đó) | "Bài này chẳng giúp gì cho câu hỏi" |

## Quy trình 3 bước cho MỖI câu hỏi

1. `search("...")` — dùng **câu hỏi đầy đủ cũng được**, công cụ tự tách từ
2. `show("<doc_id>")` — đọc toàn văn bài còn phân vân
3. `judge(...)` — chấm điểm

Cuối cùng: `save()` → ghi ra `eval/qrels.csv`.

In [1]:
# ── Khởi tạo: nạp corpus + chỉ mục (chạy 1 lần, ~5-10 giây) ────
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from explore import load_docs, build_index, search as _search   # dùng CHUNG code với CLI

QRELS = ROOT / "eval" / "qrels.csv"
_docs, SOURCE = load_docs()
INDEX = build_index(_docs)
docs = pd.DataFrame(_docs)
print(f"Corpus: {len(docs)} bài · {docs['chars'].sum()/1e6:.2f} MB · nguồn: {SOURCE}")

Corpus: 179 bài · 2.47 MB · nguồn: curated (data\processed\curated.jsonl)


In [2]:
# ── Công cụ chấm điểm (không cần sửa) ──────────────────────────
def search(query: str, k: int = 8):
    """Xếp hạng bài theo truy vấn — dùng câu hỏi đầy đủ hoặc vài từ khoá đều được."""
    rows = _search(query, _docs, k=k, index=INDEX)
    if not rows:
        return f"Không tìm thấy bài nào cho '{query}'"
    return pd.DataFrame(rows)[["doc_id", "title", "chars", "matched", "score", "snippet"]]

def show(doc_id, limit: int = 2000):
    """In toàn văn (hoặc `limit` ký tự đầu) một bài để chấm điểm."""
    row = docs[docs["doc_id"] == str(doc_id)]
    if row.empty:
        print(f"Không có doc_id {doc_id}"); return
    r = row.iloc[0]
    print(f"# {r['title']}\n{r['url']}\n({r['chars']:,} ký tự)\n")
    print(r["text"][:limit] + ("\n\n[... còn nữa — tăng limit để đọc thêm]" if r["chars"] > limit else ""))

# sổ chấm điểm (nạp lại nếu đã chấm từ trước)
if QRELS.exists():
    jud = pd.read_csv(QRELS, dtype={"doc_id": str}).dropna(subset=["relevance"]).to_dict("records")
    print(f"Đã nạp {len(jud)} dòng có sẵn từ {QRELS.relative_to(ROOT)}")
else:
    jud = []
    print("Chưa có file qrels — sẽ tạo khi save()")

def judge(query_id: str, query: str, grades: dict, query_class: str):
    """Chấm điểm cho MỘT câu hỏi. grades = {"doc_id": điểm, ...}

    ví dụ: judge("q001", "Học có giám sát là gì?", {"24329": 2, "19895408": 1}, "literal")
    """
    assert query_class in {"literal", "paraphrase", "multi-doc"}, "query_class: literal / paraphrase / multi-doc"
    for did, rel in grades.items():
        assert str(did) in set(docs["doc_id"]), f"doc_id {did} không tồn tại trong corpus!"
        assert rel in (0, 1, 2), "relevance chỉ nhận 0 / 1 / 2"
        jud.append({"query_id": query_id, "query": query, "doc_id": str(did),
                    "relevance": rel, "query_class": query_class})
    print(f"✓ {query_id}: chấm {len(grades)} bài ({query_class}) → tổng {len(jud)} dòng")

def progress():
    if not jud:
        print("Chưa chấm query nào."); return
    df = pd.DataFrame(jud)
    print(df.groupby(["query_id", "query_class"]).agg(số_bài=("doc_id", "count")).reset_index().to_string(index=False))
    print("\nTheo lớp:", df.groupby("query_class")["query_id"].nunique().to_dict())
    print(f"Tổng: {df['query_id'].nunique()} query · {len(df)} dòng")

def save():
    """Ghi ra eval/qrels.csv."""
    if not jud:
        print("Chưa chấm query nào — không ghi đè file qrels."); return
    df = pd.DataFrame(jud)[["query_id", "query", "doc_id", "relevance", "query_class"]]
    QRELS.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(QRELS, index=False)
    print(f"Đã lưu {len(df)} dòng → {QRELS.relative_to(ROOT)}")
print("Sẵn sàng: search() · show() · judge() · progress() · save()")

Đã nạp 12 dòng có sẵn từ eval\qrels.csv
Sẵn sàng: search() · show() · judge() · progress() · save()


## Bước 1 — Tìm bài ứng viên

Gõ **câu hỏi đầy đủ** cũng được (công cụ tự tách từ + bỏ từ dừng), hoặc gõ vài từ khoá.

> ⚠️ Công cụ này chỉ để **tìm ứng viên**, không phải để chấm điểm. Nó cố tình đơn giản
> (IDF + chuẩn hoá độ dài) — phiên bản BM25 bạn sắp code sẽ tốt hơn. Nếu kết quả xếp hạng
> chưa chuẩn, đó chính là dữ liệu quý cho phần so sánh sau này.

In [3]:
search("Học có giám sát là gì?")

,doc_id,title,chars,matched,score,snippet
0,19895408,Học tự giám sát,8010,3,210.61,Học tự giám sát (self-supervised learning hay ...
1,19893713,Học đặc trưng,24688,3,170.05,"Trong học máy, học đặc trưng (feature learning..."
2,3339820,Học sâu,57004,3,137.83,"Học sâu (tiếng Anh: deep learning, còn gọi là ..."
3,24345,Học không có giám sát,1866,3,129.94,Học không có giám sát (tiếng Anh: unsupervised...
4,20014464,Thuật ngữ trí tuệ nhân tạo,42756,3,119.03,"... đến ngành trí tuệ nhân tạo (AI), các phân ..."
5,19950271,Học vấn về AI,8233,3,107.72,Học vấn về AI hay học vấn về trí tuệ nhân tạo ...
6,24278,Học máy,19947,3,97.68,Học máy (tiếng Anh: Machine learning) là một l...
7,20012838,Tốc độ học,7071,3,91.16,"Tốc độ học (tiếng Anh: learning rate, còn được..."


## Bước 2 — Đọc bài còn phân vân

Tiêu đề đúng chủ đề → gần như chắc chắn điểm 2. Bài chỉ *nhắc tới* → phải đọc mới biết.

In [4]:
show("19895408", limit=1200)

# Học tự giám sát
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_t%E1%BB%B1_gi%C3%A1m_s%C3%A1t
(8,010 ký tự)

Học tự giám sát (self-supervised learning hay SSL) là một kỹ thuật trong học máy mà trong đó, một mô hình được huấn luyện trên một tác vụ bằng cách sử dụng chính dữ liệu để tạo ra các tín hiệu giám sát, thay vì dựa vào các nhãn bên ngoài do con người cung cấp. Trong bối cảnh của mạng thần kinh nhân tạo, học tự giám sát hướng tới việc khai thác các cấu trúc hoặc mối quan hệ nội tại trong dữ liệu đầu vào để tạo ra các tín hiệu huấn luyện có ý nghĩa. Các nhiệm vụ trong SSL được thiết kế sao cho việc giải quyết yêu cầu nắm bắt được các đặc điểm hoặc mối quan hệ quan trọng trong dữ liệu. Dữ liệu đầu vào thường được tăng cường hoặc biến đổi để tạo ra các cặp mẫu liên quan. Một mẫu sẽ đóng vai trò làm đầu vào, và mẫu còn lại được dùng để hình thành tín hiệu giám sát. Việc tăng cường này có thể bao gồm thêm nhiễu, cắt, xoay, hoặc các biến đổi khác. Học tự giám sát mô phỏng cách con người họ

## Bước 3 — Chấm điểm

Mỗi câu hỏi = 1 lệnh `judge()`, truyền dict `{doc_id: điểm}`.

Ví dụ dưới đây là **câu hỏi mẫu** — sửa theo phán đoán của CHÍNH BẠN rồi bỏ dấu `#`.

In [5]:
# judge("q001", "Học có giám sát là gì?", {
#     "24329":    2,   # bài đúng chủ đề, định nghĩa trực tiếp
#     "19895408": 1,   # "Học tự giám sát" — chủ đề khác, chỉ nhắc để so sánh
#     "20014464": 1,   # "Thuật ngữ AI" — có 1 dòng định nghĩa, không giải thích sâu
# }, "literal")

progress()

query_id query_class  số_bài
    q001     literal       3
    q002     literal       3
    q003     literal       3
    q004     literal       3

Theo lớp: {'literal': 4}
Tổng: 4 query · 12 dòng


## Checklist 12 query đợt 1

- [ ] 4 × **literal** — dùng đúng từ khoá trong bài (*"Mạng nơ-ron tích chập là gì?"*)
- [ ] 4 × **paraphrase** — KHÔNG lặp từ khoá của bài (*"Máy tính học từ ví dụ có nhãn bằng cách nào?"* thay vì *"học có giám sát"*)
- [ ] 4 × **multi-doc** — cần ≥2 bài (*"So sánh học có giám sát và học tự giám sát"* → gold phải có cả 2 bài)

**Chọn chủ đề bạn đã học** → chấm relevance tự tin hơn nhiều.
Xem 179 tiêu đề: `python scripts/explore.py --titles` (chạy ở terminal).

In [6]:
# ── Lưu lại ────────────────────────────────────────────────────
progress()
save()

query_id query_class  số_bài
    q001     literal       3
    q002     literal       3
    q003     literal       3
    q004     literal       3

Theo lớp: {'literal': 4}
Tổng: 4 query · 12 dòng
Đã lưu 12 dòng → eval\qrels.csv


In [7]:
search("Học không có giám sát là gì?")

,doc_id,title,chars,matched,score,snippet
0,19895408,Học tự giám sát,8010,3,210.61,Học tự giám sát (self-supervised learning hay ...
1,19893713,Học đặc trưng,24688,3,170.05,"Trong học máy, học đặc trưng (feature learning..."
2,3339820,Học sâu,57004,3,137.83,"Học sâu (tiếng Anh: deep learning, còn gọi là ..."
3,24345,Học không có giám sát,1866,3,129.94,Học không có giám sát (tiếng Anh: unsupervised...
4,20014464,Thuật ngữ trí tuệ nhân tạo,42756,3,119.03,"... đến ngành trí tuệ nhân tạo (AI), các phân ..."
5,19950271,Học vấn về AI,8233,3,107.72,Học vấn về AI hay học vấn về trí tuệ nhân tạo ...
6,24278,Học máy,19947,3,97.68,Học máy (tiếng Anh: Machine learning) là một l...
7,20012838,Tốc độ học,7071,3,91.16,"Tốc độ học (tiếng Anh: learning rate, còn được..."


In [8]:
show("20012838")

# Tốc độ học
https://vi.wikipedia.org/wiki/T%E1%BB%91c_%C4%91%E1%BB%99_h%E1%BB%8Dc
(7,071 ký tự)

Tốc độ học (tiếng Anh: learning rate, còn được dịch là Tỉ lệ học) là một siêu tham số (hyperparameter) trong các thuật toán tối ưu hóa như suy giảm độ dốc, giúp xác định kích thước mỗi bước cập nhật khi tham số của mô hình di chuyển về phía cực tiểu của hàm mất mát. 
Tốc độ học quá cao có thể khiến quá trình học "nhảy qua" cực tiểu cần tìm; ngược lại, tốc độ học quá thấp khiến quá trình hội tụ chậm hoặc mắc kẹt tại một điểm cực tiểu địa phương không mong muốn. Để tránh các vấn đề trên, tốc độ học có thể được thay đổi trong quá trình huấn luyện theo một lịch trình định trước (learning rate schedule) hoặc bằng một tốc độ học thích ứng (adaptive learning rate). 
Trong các tài liệu về điều khiển thích nghi (adaptive control) và xấp xỉ ngẫu nhiên, đại lượng tương tự còn được gọi là hệ số khuếch đại (gain).


== Thay đổi tốc độ học trong quá trình huấn luyện ==
Tốc độ học khởi tạo có thể giữ một

In [9]:
show("24345")

# Học không có giám sát
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_kh%C3%B4ng_c%C3%B3_gi%C3%A1m_s%C3%A1t
(1,866 ký tự)

Học không có giám sát (tiếng Anh: unsupervised learning) là một phương pháp của ngành học máy nhằm tìm ra một mô hình mà phù hợp với các quan sát. Nó khác biệt với học có giám sát ở chỗ là đầu ra đúng tương ứng cho mỗi đầu vào là không biết trước. Trong học không có giám sát, một tập dữ liệu đầu vào được thu thập. Học không có giám sát thường đối xử với các đối tượng đầu vào như là một tập các biến ngẫu nhiên. Sau đó, một mô hình mật độ kết hợp sẽ được xây dựng cho tập dữ liệu đó.
Học không có giám sát có thể được dùng kết hợp với suy luận Bayes để cho ra xác suất có điều kiện (nghĩa là học có giám sát) cho bất kì biến ngẫu nhiên nào khi biết trước các biến khác.
Học không có giám sát cũng hữu ích cho việc nén dữ liệu: về cơ bản, mọi giải thuật nén dữ liệu hoặc là dựa vào một phân bố xác suất trên một tập đầu vào một cách tường minh hay không tường minh.
Một dạng khác 

In [10]:
show("19893713")

# Học đặc trưng
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_%C4%91%E1%BA%B7c_tr%C6%B0ng
(24,688 ký tự)

Trong học máy, học đặc trưng (feature learning) hay học biểu diễn (representation learning) là tập hợp các kỹ thuật cho phép hệ thống tự động khám phá các biểu diễn cần thiết cho việc phát hiện hoặc phân loại đặc trưng từ dữ liệu thô. Điều này thay thế cho việc xây dựng đặc trưng (feature engineering) thủ công và cho phép máy học cả đặc trưng lẫn thực hiện nhiệm vụ cụ thể.
Học đặc trưng được thúc đẩy bởi thực tế rằng các bài toán trong học máy như phân loại thường yêu cầu đầu vào dễ dàng và thuận tiện cho việc xử lý toán học và tính toán. Tuy nhiên, dữ liệu thực tế như hình ảnh, video và dữ liệu từ cảm biến không thể dễ dàng xác định các đặc trưng cụ thể thông qua các thuật toán rõ ràng. Một phương pháp thay thế là khám phá các đặc trưng hoặc biểu diễn này thông qua quá trình quan sát, mà không phụ thuộc vào các thuật toán cụ thể.
Học đặc trưng có thể là có giám sát, không giám sát hoặ

In [11]:
judge("q001", "Học không có giám sát là gì?", {
    "24345": 2, "19893713": 1, "20012838": 0
}, "literal")

progress()

✓ q001: chấm 3 bài (literal) → tổng 15 dòng
query_id query_class  số_bài
    q001     literal       6
    q002     literal       3
    q003     literal       3
    q004     literal       3

Theo lớp: {'literal': 4}
Tổng: 4 query · 15 dòng


In [12]:
search("Phân loại nhị phân là gì?")

,doc_id,title,chars,matched,score,snippet
0,16581586,Phân kỳ Kullback–Leibler,74966,4,269.28,"Phân kỳ Kullback–Leibler (viết tắt KL, còn gọi..."
1,20023680,Phân loại đa lớp,24460,4,200.97,"Trong học máy và phân loại thống kê, phân loại..."
2,60027,Tổng hợp giọng nói,49950,4,151.60,...hệ thống tổng hợp giọng nói nào cũng có đầu...
3,30712,Phân loại nhị phân,4112,4,139.46,Phân loại nhị phân (tiếng Anh: Binary classifi...
4,20014464,Thuật ngữ trí tuệ nhân tạo,42756,4,132.30,...và khái niệm liên quan đến ngành trí tuệ nh...
5,20015375,Entropy chéo,24899,4,121.52,...ng trong lý thuyết thông tin đo lường sự kh...
6,19893713,Học đặc trưng,24688,4,120.92,...ng khám phá các biểu diễn cần thiết cho việ...
7,19991608,Perceptron,25201,4,96.78,"..., perceptron là một thuật toán học có giám ..."


In [13]:
show("30712")

# Phân loại nhị phân
https://vi.wikipedia.org/wiki/Ph%C3%A2n_lo%E1%BA%A1i_nh%E1%BB%8B_ph%C3%A2n
(4,112 ký tự)

Phân loại nhị phân (tiếng Anh: Binary classification) là nhiệm vụ phận loại các phần tử của một tập hợp các đối tượng ra thành 2 nhóm dựa trên cơ sở là chúng có một thuộc tính nào đó hay không (hay còn gọi là tiêu chí). Một số nhiệm vụ phân loại nhị phân điển hình:

kiểm tra y khoa xem một bệnh nhân có bệnh nào đó hay không (thuộc tính để phân loại là căn bệnh đó)
quản lý chất lượng trong nhà máy, ví dụ: xác định xem một sản phẩm làm ra là đủ tốt để bán chưa, hay nên loại bỏ nó (thuộc tính để phân loại là tính đủ tốt)
xác định xem một trang hay một bài báo có nên nằm trong tập kết quả của một truy vấn hay không (thuộc tính là độ liên quan của bài báo - thường là sự hiện diện của một số từ nào đó trong bài báo đó)
Phân loại nói chung là một trong những vấn đề được nghiên cứu trong khoa học máy tính với mục đích học tự động các hệ thống phân loại. Một số phương pháp thích hợp ch

In [14]:
show("19893713")

# Học đặc trưng
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_%C4%91%E1%BA%B7c_tr%C6%B0ng
(24,688 ký tự)

Trong học máy, học đặc trưng (feature learning) hay học biểu diễn (representation learning) là tập hợp các kỹ thuật cho phép hệ thống tự động khám phá các biểu diễn cần thiết cho việc phát hiện hoặc phân loại đặc trưng từ dữ liệu thô. Điều này thay thế cho việc xây dựng đặc trưng (feature engineering) thủ công và cho phép máy học cả đặc trưng lẫn thực hiện nhiệm vụ cụ thể.
Học đặc trưng được thúc đẩy bởi thực tế rằng các bài toán trong học máy như phân loại thường yêu cầu đầu vào dễ dàng và thuận tiện cho việc xử lý toán học và tính toán. Tuy nhiên, dữ liệu thực tế như hình ảnh, video và dữ liệu từ cảm biến không thể dễ dàng xác định các đặc trưng cụ thể thông qua các thuật toán rõ ràng. Một phương pháp thay thế là khám phá các đặc trưng hoặc biểu diễn này thông qua quá trình quan sát, mà không phụ thuộc vào các thuật toán cụ thể.
Học đặc trưng có thể là có giám sát, không giám sát hoặ

In [15]:
show("20023680")

# Phân loại đa lớp
https://vi.wikipedia.org/wiki/Ph%C3%A2n_lo%E1%BA%A1i_%C4%91a_l%E1%BB%9Bp
(24,460 ký tự)

Trong học máy và phân loại thống kê, phân loại đa lớp (tiếng Anh: multiclass classification hay multinomial classification) là bài toán phân loại các đối tượng quan sát vào một trong ba hoặc nhiều lớp phân biệt (phân loại đối tượng vào đúng một trong hai lớp được gọi là phân loại nhị phân).
Ví dụ, việc xác định một hình ảnh chụp quả chuối, quả đào, quả cam hay quả táo là một bài toán phân loại đa lớp với 4 lớp xuất ra có thể xảy ra. Trong khi đó, việc xác định xem một hình ảnh có chứa quả táo hay không lại là bài toán phân loại nhị phân (với hai lớp: "có táo" và "không có táo").
Cần phân biệt rõ phân loại đa lớp với phân loại đa nhãn (multi-label classification) — nơi mỗi đối tượng có thể được gán đồng thời nhiều nhãn (ví dụ: một bức ảnh vừa có nhãn "quả táo" vừa có nhãn "quả cam") — và phân loại một lớp (one-class classification) nhằm phát hiện các điểm bất thường.


== Tổng qua

In [16]:
judge("q002", "Phân loại nhị phân là gì?", {
    "30712": 2, "20023680": 1, "19893713": 0
}, "literal")

progress()

✓ q002: chấm 3 bài (literal) → tổng 18 dòng
query_id query_class  số_bài
    q001     literal       6
    q002     literal       6
    q003     literal       3
    q004     literal       3

Theo lớp: {'literal': 4}
Tổng: 4 query · 18 dòng


In [17]:
search("Chưng cất chi thức là gì?")

,doc_id,title,chars,matched,score,snippet
0,19917007,Chưng cất tri thức,42272,3,108.30,"Trong học máy, chưng cất tri thức (knowledge d..."
1,25735,Lịch sử ngành trí tuệ nhân tạo,97944,3,97.82,...ủa máy tính điện tử vào thập niên 1940. Khi...
2,20017867,Thung lũng kỳ lạ,27899,3,72.54,...cho con người. Giả thuyết thung lũng kỳ lạ ...
3,19995683,Mùa đông AI,26218,3,68.62,...g và nhà đầu tư. Hệ quả là nhiều chương trì...
4,19988121,Học liên kết,37355,3,58.08,"...c bộ của từng người dùng, nơi các phép tính..."
5,20035026,Thành viên:Qesefe/Quên thảm họa,25609,3,47.39,Quên thảm họa (tiếng Anh: catastrophic forgett...
6,20014464,Thuật ngữ trí tuệ nhân tạo,42756,3,44.24,"...êu nhân ở cờ vua, shogi, và cờ vây mà không..."
7,19989487,Machine unlearning,18327,3,43.07,...(tạm dịch: Máy học cách quên) là một kỹ thu...


In [18]:
show("19917007")

# Chưng cất tri thức
https://vi.wikipedia.org/wiki/Ch%C6%B0ng_c%E1%BA%A5t_tri_th%E1%BB%A9c
(42,272 ký tự)

Trong học máy, chưng cất tri thức (knowledge distillation) hay chưng cất mô hình (model distillation) là quá trình chuyển giao tri thức từ mô hình lớn sang mô hình nhỏ hơn. Dù các mô hình lớn (như mạng nơ-ron sâu hoặc tập hợp nhiều mô hình) có khả năng lưu trữ tri thức cao hơn, khả năng này có thể chưa được sử dụng hết. Chi phí tính toán để chạy mô hình vẫn cao ngay cả khi nó chỉ sử dụng một phần nhỏ tri thức. Chưng cất tri thức cho phép chuyển giao kiến thức từ mô hình lớn sang nhỏ mà không làm giảm hiệu quả. Các mô hình nhỏ tốn ít chi phí tính toán hơn, có thể triển khai trên thiết bị phần cứng yếu hơn (như điện thoại di động).
Khái niệm này khác với nén mô hình (model compression) - phương pháp giảm kích thước mô hình lớn mà không cần huấn luyện mô hình mới. Nén mô hình thường giữ nguyên kiến trúc và số lượng tham số, chỉ giảm số bit trên mỗi tham số.
Chưng cất tri thức đã được

In [19]:
show("25735")

# Lịch sử ngành trí tuệ nhân tạo
https://vi.wikipedia.org/wiki/L%E1%BB%8Bch_s%E1%BB%AD_ng%C3%A0nh_tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o
(97,944 ký tự)

Lịch sử ngành trí tuệ nhân tạo (AI) có những tiền đề từ thời cổ đại, khi con người kể các câu chuyện về những sinh vật nhân tạo và cỗ máy được ban cho trí thông minh. Qua nhiều thế kỷ, sự phát triển của logic, toán học và các phương pháp suy luận dựa trên quy tắc đã góp phần dẫn đến sự ra đời của máy tính điện tử vào thập niên 1940. Khi máy tính có thể thực hiện nhiều chuỗi thao tác khác nhau theo chương trình, các nhà khoa học bắt đầu nghiêm túc xem xét khả năng chế tạo một "bộ não điện tử", tức một hệ thống có thể mô phỏng một số năng lực trí tuệ của con người.
Trí tuệ nhân tạo chính thức hình thành như một lĩnh vực nghiên cứu tại một hội thảo tổ chức ở Đại học Dartmouth năm 1956. Tại đây, Allen Newell và Herbert A. Simon, với sự hỗ trợ của J. C. Shaw, trình bày Logic Theorist, một trong những chương trình AI đầu tiên. Newell về 

In [20]:
show("19989487")

# Machine unlearning
https://vi.wikipedia.org/wiki/Machine_unlearning
(18,327 ký tự)

Trong học máy, Machine unlearning (tạm dịch: Máy học cách quên) là một kỹ thuật giúp xóa kiến thức từ một phần dữ liệu đã học khỏi mô hình. Với sự phổ biến của các mô hình ngôn ngữ lớn như ChatGPT hay Gemini, nhiều người cho rằng chỉ cần yêu cầu mô hình "hãy quên đi" hay xóa đoạn chat là có thể khiến chúng "quên" những gì mình mong muốn. Tuy nhiên, trên thực tế, điều này không hề thay đổi tham số của mô hình mạng thần kinh nhân tạo, và chẳng mấy ảnh hưởng đến việc chúng có thực sự quên đi hay không.
Việc mô hình có thực sự "quên" đi hay không thường được định nghĩa theo "tiêu chuẩn vàng" là khiến mô hình phải học lại tất cả các kiến thức từ đầu theo đúng trình tự nó đã từng được huấn luyện, với điểm khác biệt duy nhất là không bao gồm phần dữ liệu mình muốn quên đi kia. Đây được gọi là phương pháp Học-lại-từ-đầu (Retrain-from-scratch), một giải pháp hết sức tốn kém nhưng đảm bảo được mô hình chắc chắn

In [21]:
judge("q003", "Chưng cất chi thức là gì?", {
    "19917007": 2, "19989487": 0, "25735": 0
}, "literal")

progress()

✓ q003: chấm 3 bài (literal) → tổng 21 dòng
query_id query_class  số_bài
    q001     literal       6
    q002     literal       6
    q003     literal       6
    q004     literal       3

Theo lớp: {'literal': 4}
Tổng: 4 query · 21 dòng


In [22]:
search("Học máy lượng tử là gì?")

,doc_id,title,chars,matched,score,snippet
0,20015836,Học máy lượng tử,4150,3,188.17,Học máy lượng tử (Quantum machine learning - Q...
1,20014464,Thuật ngữ trí tuệ nhân tạo,42756,3,147.66,"... đến ngành trí tuệ nhân tạo (AI), các phân ..."
2,24278,Học máy,19947,3,124.35,Học máy (tiếng Anh: Machine learning) là một l...
3,25735,Lịch sử ngành trí tuệ nhân tạo,97944,3,117.28,...on người kể các câu chuyện về những sinh vậ...
4,19988121,Học liên kết,37355,3,111.16,Học liên kết (tiếng Anh: federated learning) l...
5,3339820,Học sâu,57004,3,106.04,"Học sâu (tiếng Anh: deep learning, còn gọi là ..."
6,19950271,Học vấn về AI,8233,3,99.73,Học vấn về AI hay học vấn về trí tuệ nhân tạo ...
7,581452,Phép thử Turing,23683,3,89.59,...ing test) do Alan Turing đề xuất nhằm đánh ...


In [23]:
show("20015836")

# Học máy lượng tử
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_m%C3%A1y_l%C6%B0%E1%BB%A3ng_t%E1%BB%AD
(4,150 ký tự)

Học máy lượng tử (Quantum machine learning - QML) là lĩnh vực nghiên cứu các thuật toán lượng tử dành cho học máy. Học máy lượng tử thường đề cập đến các thuật toán lượng tử dành cho các tác vụ học máy phân tích dữ liệu cổ điển, đôi khi còn được gọi là học máy tăng cường lượng tử (Quantum-enhanced machine learning). Thuật ngữ "học máy lượng tử" đôi khi được sử dụng để chỉ các phương pháp học máy cổ điển áp dụng cho dữ liệu được tạo ra từ các thí nghiệm lượng tử (tức là học máy của các hệ lượng tử), chẳng hạn như việc học tập các Chuyển pha của một hệ lượng tử hoặc việc tạo ra các thí nghiệm lượng tử mới. Các thuật toán QML sử dụng các Qubit và các phép toán lượng tử để cố gắng cải thiện độ phức tạp không gian và độ phức tạp thời gian của các thuật toán học máy cổ điển. 
Các phương pháp QML lai (hybrid QML methods) bao gồm cả việc xử lý cổ điển và lượng tử, trong đó các chư

In [24]:
show("25735")

# Lịch sử ngành trí tuệ nhân tạo
https://vi.wikipedia.org/wiki/L%E1%BB%8Bch_s%E1%BB%AD_ng%C3%A0nh_tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o
(97,944 ký tự)

Lịch sử ngành trí tuệ nhân tạo (AI) có những tiền đề từ thời cổ đại, khi con người kể các câu chuyện về những sinh vật nhân tạo và cỗ máy được ban cho trí thông minh. Qua nhiều thế kỷ, sự phát triển của logic, toán học và các phương pháp suy luận dựa trên quy tắc đã góp phần dẫn đến sự ra đời của máy tính điện tử vào thập niên 1940. Khi máy tính có thể thực hiện nhiều chuỗi thao tác khác nhau theo chương trình, các nhà khoa học bắt đầu nghiêm túc xem xét khả năng chế tạo một "bộ não điện tử", tức một hệ thống có thể mô phỏng một số năng lực trí tuệ của con người.
Trí tuệ nhân tạo chính thức hình thành như một lĩnh vực nghiên cứu tại một hội thảo tổ chức ở Đại học Dartmouth năm 1956. Tại đây, Allen Newell và Herbert A. Simon, với sự hỗ trợ của J. C. Shaw, trình bày Logic Theorist, một trong những chương trình AI đầu tiên. Newell về 

In [25]:
show("19950271")

# Học vấn về AI
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_v%E1%BA%A5n_v%E1%BB%81_AI
(8,233 ký tự)

Học vấn về AI hay học vấn về trí tuệ nhân tạo (tiếng Anh: AI literacy) là khả năng hiểu, sử dụng, giám sát và suy ngẫm một cách có phê phán về các ứng dụng AI. Thuật ngữ này thường dùng để chỉ việc giảng dạy các kỹ năng và kiến thức cho công chúng, đặc biệt là những người không thành thạo về AI.
Một số người cho rằng học vấn về AI là rất cần thiết cho học sinh và sinh viên. Ngược lại, một số giáo sư lại cấm sử dụng AI trong lớp học và trong tất cả các bài tập. Họ còn đưa ra những hình phạt nghiêm khắc cho việc sử dụng AI, coi đó là một hành vi gian lận. AI được sử dụng trong nhiều ứng dụng khác nhau, ví dụ như ô tô tự lái, trợ lý ảo và các mô hình AI tạo sinh có khả năng tạo văn bản. Người dùng các công cụ này cần có khả năng đưa ra quyết định một cách sáng suốt. Học vấn về AI cũng có thể ảnh hưởng đến cơ hội việc làm trong tương lai của sinh viên.


== Định nghĩa ==
Một trong những định 

In [26]:
judge("q004", "Học máy lượng tử là gì", {
    "20015836": 2, "19950271": 0, "25735": 0
}, "literal")

progress()

✓ q004: chấm 3 bài (literal) → tổng 24 dòng
query_id query_class  số_bài
    q001     literal       6
    q002     literal       6
    q003     literal       6
    q004     literal       6

Theo lớp: {'literal': 4}
Tổng: 4 query · 24 dòng


In [27]:
save()

Đã lưu 24 dòng → eval\qrels.csv


In [28]:
show("24278")

# Học máy
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_m%C3%A1y
(19,947 ký tự)

Học máy (tiếng Anh: Machine learning) là một lĩnh vực của trí tuệ nhân tạo (AI), hướng tới việc xây dựng các thuật toán và mô hình có khả năng "học" từ dữ liệu huấn luyện. Từ đó, các mô hình này có thể đưa ra dự đoán hoặc quyết định mà không cần được lập trình chi tiết cho từng tình huống cụ thể. Gần đây, một nhánh nhỏ của học máy là học sâu đã sử dụng mạng thần kinh nhân tạo giúp đạt kết quả vượt trội so với các phương pháp học máy khác.
Học máy có liên hệ mật thiết với thống kê, vì cả hai lĩnh vực đều xây dựng mô hình từ dữ liệu để đưa ra suy luận. Điểm khác biệt chính là thống kê truyền thống thường tập trung vào việc kiểm định giả thuyết và diễn giải kết quả, trong khi học máy ưu tiên khả năng dự đoán và tự động hóa trên dữ liệu lớn.
Ngày nay, học máy được ứng dụng rộng rãi trong nhiều lĩnh vực, từ tìm kiếm thông tin, chẩn đoán y khoa, phát hiện thư rác và gian lận tài chính, đến nhận dạng tiếng nói, dịch 